# ⚽ Sports Analytics Tool — Google Colab Runner

This notebook downloads the Sports Analytics Tool and launches it with a public URL via **pyngrok**.  
Run all cells top-to-bottom. The public URL appears at the end of **Cell 3**.

> **Requirements:** Google account, internet connection.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q streamlit pyngrok pandas numpy plotly scikit-learn scipy matplotlib seaborn

In [ ]:
# Cell 2 — Upload the sports_analytics_tool.zip
# Either upload a zip of the tool or paste the code directly.
from google.colab import files
import zipfile, os

print('Upload sports_analytics_tool.zip ...')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f'Extracted {fname}')

# Confirm structure
!ls sports_analytics_tool/

In [ ]:
# Cell 3 — Launch Streamlit with ngrok tunnel
import subprocess, threading, time
from pyngrok import ngrok, conf

# Optional: set your ngrok auth token for longer sessions
# ngrok.set_auth_token('YOUR_TOKEN_HERE')

os.chdir('sports_analytics_tool')

# Start Streamlit in background thread
def run_streamlit():
    subprocess.run(
        ['streamlit', 'run', 'app.py',
         '--server.port=8501',
         '--server.headless=true',
         '--browser.gatherUsageStats=false'],
        check=True
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)  # wait for Streamlit to start

# Open tunnel
tunnel = ngrok.connect(8501)
print('\n' + '='*60)
print(f'🚀 Sports Analytics Tool is live at:')
print(f'   {tunnel.public_url}')
print('='*60)
print('Keep this cell running. Close the tab to stop.')

---
## Standalone Analysis (no Streamlit)

Run the cells below to explore the analytics directly in the notebook without launching the UI.

In [ ]:
import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from utils.data_generator import generate_players, generate_training_sessions, generate_matches
from utils.analytics import calculate_acwr, cluster_players, predict_injury_risk

players_df   = generate_players()
training_df  = generate_training_sessions(players_df)
matches_df, mp_df, events_df = generate_matches(players_df)

print(f'Players: {len(players_df)}')
print(f'Training sessions: {len(training_df)}')
print(f'Matches: {len(matches_df)}')
print(f'Match player rows: {len(mp_df)}')
training_df.head()

In [ ]:
# ACWR example for player 1
p1 = training_df[training_df['player_id'] == 1]
acwr_df = calculate_acwr(p1['session_rpe'], p1['date'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=acwr_df['date'], y=acwr_df['acwr'],
                         mode='lines', name='ACWR',
                         line=dict(color='#64ffda', width=2)))
fig.add_hline(y=1.3, line_dash='dash', line_color='orange', annotation_text='1.3 Caution')
fig.add_hline(y=1.5, line_dash='dash', line_color='red',    annotation_text='1.5 High Risk')
fig.update_layout(title='ACWR — James Kellerman (GK)', template='plotly_dark', height=350)
fig.show()

In [ ]:
# K-means clustering
cluster_df = cluster_players(mp_df, n_clusters=5)

fig = px.scatter(cluster_df, x='pca_x', y='pca_y',
                 color='cluster_label', symbol='position',
                 hover_data=['player_name'],
                 template='plotly_dark', height=450,
                 title='Player Clustering — PCA of Match Performance')
fig.update_traces(marker_size=12)
fig.show()

In [ ]:
# xG shot map for match 1
shots_m1 = events_df[(events_df['match_id'] == 1) & (events_df['event_type'] == 'shot')]

fig = px.scatter(shots_m1, x='x', y='y',
                 size='xg', color='goal',
                 hover_data=['player_name', 'minute', 'xg'],
                 color_discrete_map={True: '#64ffda', False: '#ff6b6b'},
                 template='plotly_dark', height=420,
                 title='Shot Map — Match 1')
fig.update_layout(xaxis_range=[0, 110], yaxis_range=[0, 68],
                  xaxis_title='Pitch Length (m)', yaxis_title='Pitch Width (m)')
fig.show()